# GovBench-Med — Full Experiment Run
**Before running:** Runtime → Change runtime type → **T4 GPU**

Run cells top to bottom.

In [ ]:
# ── CELL 1: Install zstd + Ollama, start server, pull models ──
import subprocess, threading, time, sys

# 1a. Install zstd (required by Ollama installer)
subprocess.run(['apt-get', 'install', '-y', '-q', 'zstd'], check=True)
print('zstd installed')

# 1b. Install Ollama
r = subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, capture_output=True, text=True)
print(r.stdout[-300:] if r.stdout else '')
if r.returncode != 0:
    print('INSTALL STDERR:', r.stderr[-300:])

# 1c. Start Ollama server in background thread
def _serve():
    subprocess.run(['ollama', 'serve'], capture_output=True)
threading.Thread(target=_serve, daemon=True).start()

# 1d. Wait until server is ready (poll /api/tags)
import urllib.request, json
for attempt in range(30):
    time.sleep(2)
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3)
        print(f'Ollama ready after {(attempt+1)*2}s')
        break
    except Exception:
        print(f'  waiting... ({(attempt+1)*2}s)', end='\r')
else:
    raise RuntimeError('Ollama server did not start in 60s')

# 1e. Pull the three models (runs sequentially, ~5 min each on T4)
for model in ['llama3.1:8b', 'mistral:7b', 'qwen2.5:7b']:
    print(f'Pulling {model} ...')
    r = subprocess.run(['ollama', 'pull', model],
                       capture_output=True, text=True, timeout=900)
    if r.returncode == 0:
        print(f'  ✓ {model}')
    else:
        print(f'  ✗ {model}: {r.stderr[:200]}')

print('\nAll models ready.')

In [ ]:
# ── CELL 2: Clone repo  ──
# Push your local project to GitHub first, then paste the URL below.
# If you haven't pushed yet, run the MANUAL UPLOAD alternative (see comment).
import os, subprocess

REPO_URL = 'https://github.com/YOUR_USERNAME/govbench-med.git'  # <-- UPDATE

if REPO_URL == 'https://github.com/YOUR_USERNAME/govbench-med.git':
    raise ValueError('Update REPO_URL with your actual GitHub repo URL!')

if not os.path.exists('govbench-med'):
    subprocess.run(['git', 'clone', REPO_URL], check=True)
else:
    subprocess.run(['git', '-C', 'govbench-med', 'pull'])

os.chdir('govbench-med')
print('Working dir:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── CELL 2b (ALTERNATIVE): Upload a zip instead of GitHub ──
# Use this if you haven't set up GitHub yet.
# 1. On your PC: zip the govbench-med folder
# 2. Uncomment and run this cell, upload when prompted

# from google.colab import files
# import zipfile, os
# uploaded = files.upload()          # upload govbench-med.zip
# fname = list(uploaded.keys())[0]
# with zipfile.ZipFile(fname, 'r') as z:
#     z.extractall('.')
# os.chdir('govbench-med')
# print('Extracted to:', os.getcwd())

In [ ]:
# ── CELL 3: Install Python deps ──
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'requests', 'pandas', 'numpy', 'matplotlib',
    'seaborn', 'scipy', 'scikit-learn', 'tqdm'], check=True)
print('Done')

In [ ]:
# ── CELL 4: Download real datasets ──
import subprocess, sys
r = subprocess.run([sys.executable, 'scripts/prepare_data.py'],
                   capture_output=True, text=True, timeout=300)
print(r.stdout)
if r.returncode != 0:
    print('ERR:', r.stderr[-500:])

In [ ]:
# ── CELL 5: Speed check (should be 3-6s on T4 GPU) ──
import time, requests as req
t0 = time.time()
r = req.post('http://localhost:11434/api/generate',
    json={'model': 'llama3.1:8b',
          'prompt': 'Patient: 45yo male, chest pain, diaphoresis. Diagnosis?',
          'stream': False},
    timeout=60)
elapsed = time.time() - t0
print(f'Latency: {elapsed:.1f}s')
print(f'Response: {r.json()["response"][:120]}')
if elapsed < 15:
    print('GPU is working correctly.')
else:
    print('WARNING: Running on CPU. Consider switching to T4 GPU runtime.')

In [ ]:
# ── CELL 6: Run experiments ──
# Start with the quick run to verify everything, then uncomment the full run.
import subprocess, sys

# QUICK RUN: 50 cases × G0+G1+G2 × llama only × 1 seed (~25 min on T4)
# Good for verifying everything works before committing to the full run.
r = subprocess.run([
    sys.executable, 'scripts/run_experiments.py',
    '--n', '50',
    '--level', 'G0',
    '--level', 'G1',
    '--level', 'G2',
    '--model', 'llama3.1:8b',
    '--seed', '42'
], capture_output=False, timeout=7200)  # 2hr timeout

# FULL RUN (uncomment after quick run succeeds — runs overnight):
# r = subprocess.run([sys.executable, 'scripts/run_experiments.py', '--full'],
#                    capture_output=False, timeout=86400)

In [ ]:
# ── CELL 7: Pareto frontier plots ──
import pandas as pd, matplotlib.pyplot as plt, glob, os

csvs = sorted(glob.glob('experiments/results/results_*.csv'))
assert csvs, 'No results found — run Cell 6 first'
df = pd.read_csv(csvs[-1])
print(f'{len(df)} rows from {csvs[-1]}')

# Aggregate
agg = df.groupby(['model', 'governance_level']).agg(
    css=('css', 'mean'),
    cmr=('critical_miss', 'mean'),
    hir=('hallucination_impactful', 'mean'),
    urr=('unsafe_reassurance', 'mean'),
    acc=('top1_correct', 'mean'),
    tokens=('total_tokens', 'mean'),
    latency=('total_latency', 'mean'),
).reset_index()

# Normalise cost relative to G0 per model
for model in agg['model'].unique():
    g0_tok = agg[(agg['model']==model) & (agg['governance_level']=='G0')]['tokens'].values
    g0_lat = agg[(agg['model']==model) & (agg['governance_level']=='G0')]['latency'].values
    if len(g0_tok) == 0:
        continue
    mask = agg['model'] == model
    agg.loc[mask, 'tc_norm'] = agg.loc[mask, 'tokens'] / g0_tok[0]
    agg.loc[mask, 'lat_norm'] = agg.loc[mask, 'latency'] / g0_lat[0]
agg['ccs'] = 0.6 * agg['tc_norm'] + 0.4 * agg['lat_norm']

# ── Figure 1: Pareto frontier ──
fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#2196F3', '#FF5722', '#4CAF50']
level_order = ['G0','G1','G2','G3','G4']
for i, model in enumerate(sorted(agg['model'].unique())):
    mdf = agg[agg['model']==model].copy()
    mdf['level_n'] = mdf['governance_level'].map({l:j for j,l in enumerate(level_order)})
    mdf = mdf.sort_values('level_n')
    ax.plot(mdf['ccs'], mdf['css'], 'o-', color=colors[i % len(colors)],
            label=model.split(':')[0], linewidth=2.5, markersize=9)
    for _, row in mdf.iterrows():
        ax.annotate(row['governance_level'], (row['ccs'], row['css']),
                    textcoords='offset points', xytext=(6, 4), fontsize=9)
ax.set_xlabel('Composite Cost Score (CCS) — normalised to G0', fontsize=12)
ax.set_ylabel('Clinical Safety Score (CSS)', fontsize=12)
ax.set_title('GovBench-Med: Governance–Cost Pareto Frontier', fontsize=13)
ax.legend(); ax.grid(True, alpha=0.3)
os.makedirs('paper/figures', exist_ok=True)
fig.tight_layout()
fig.savefig('paper/figures/pareto_frontier.png', dpi=300)
plt.show()

# ── Figure 2: Safety breakdown heatmap ──
import seaborn as sns
pivot = agg.pivot(index='governance_level', columns='model', values='css')
pivot = pivot.reindex([l for l in level_order if l in pivot.index])
fig2, ax2 = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn', ax=ax2,
            vmin=0.5, vmax=1.0, linewidths=0.5)
ax2.set_title('Clinical Safety Score by Governance Level and Model')
fig2.tight_layout()
fig2.savefig('paper/figures/css_heatmap.png', dpi=300)
plt.show()

print('\n--- Aggregate Table ---')
print(agg[['model','governance_level','css','cmr','hir','urr','acc','ccs']].to_string(index=False))

In [ ]:
# ── CELL 8: Governance Efficiency (GE) table ──
# GE = ΔCSS / ΔCCS  (safety gain per unit of extra cost)
level_order = ['G0','G1','G2','G3','G4']
print(f'{"Model":<20} {"Transition":<12} {"ΔCSS":>8} {"ΔCCS":>8} {"GE":>8}')
print('-' * 60)
for model in sorted(agg['model'].unique()):
    mdf = agg[agg['model']==model].set_index('governance_level')
    for i in range(len(level_order)-1):
        l0, l1 = level_order[i], level_order[i+1]
        if l0 not in mdf.index or l1 not in mdf.index:
            continue
        dcss = mdf.loc[l1,'css'] - mdf.loc[l0,'css']
        dccs = mdf.loc[l1,'ccs'] - mdf.loc[l0,'ccs']
        ge = dcss / dccs if abs(dccs) > 1e-6 else float('inf')
        flag = '← knee' if 0 < ge < 1 else ('← efficient' if ge >= 1 else '')
        print(f'{model.split(":")[0]:<20} {l0}→{l1:<8} {dcss:>8.4f} {dccs:>8.4f} {ge:>8.4f} {flag}')
    print()

In [ ]:
# ── CELL 9: Download all results as zip ──
from google.colab import files
import glob, zipfile

with zipfile.ZipFile('/content/govbench_results.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in glob.glob('experiments/results/*.csv'):
        zf.write(f)
    for f in glob.glob('experiments/results/*.json'):
        zf.write(f)
    for f in glob.glob('paper/figures/*.png'):
        zf.write(f)

print('Downloading...')
files.download('/content/govbench_results.zip')